# 05 — Person-Centered Profile Analysis (Gaussian Mixture)

**Input:** `composite_scores.csv` from notebook 3.

A second, independent lens on the same question: instead of a regression, group
students by their AIU/CO/AIL/MMR scores (CTE deliberately excluded from
grouping) and check whether the resulting profiles differ in CTE.

**Method note:** profiles are estimated with `sklearn.mixture.GaussianMixture`,
a finite Gaussian mixture model — a close relative of, but not identical to,
Mplus-style latent profile analysis. Referred to here as Gaussian-mixture
profile analysis for precision.

**On choosing k:** BIC and entropy alone are not sufficient to pick the number
of profiles — a lower-BIC solution can still be the wrong choice if it splits
an existing profile into two groups that do not actually differ on the outcome
of interest. This notebook fits k=3 and k=4, checks profile sizes, checks
whether k=4 adds a real distinction or just fragments an existing profile (via
Tukey HSD and a k=3-vs-k=4 cross-tabulation), and picks based on that evidence
rather than BIC alone.

Requires `scikit-learn` and `statsmodels`.

See `../results/profile_model.json` for the fitted model's exported parameters
(means, covariances, weights, standardization constants) and a verified
classification formula for applying this model to new respondents.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

comp = pd.read_csv("composite_scores.csv", index_col=0)
indicators = ["AIU", "CO", "AIL", "MMR"]
Z = (comp[indicators] - comp[indicators].mean()) / comp[indicators].std(ddof=1)
print("N =", len(comp))

## Step 1 — BIC and entropy across candidate k

In [ ]:
for k in range(2, 5):
    gm = GaussianMixture(n_components=k, covariance_type="diag", n_init=20, random_state=42)
    gm.fit(Z.values)
    bic = gm.bic(Z.values)
    post = gm.predict_proba(Z.values)
    entropy = 1 - (-(post * np.log(post + 1e-12)).sum() / (len(Z) * np.log(k)))
    print(f"k={k}: BIC={bic:.1f}  entropy={entropy:.3f}")

## Step 2 — Fit both k=3 and k=4; do not pick yet

BIC and entropy both favor k=4 here — that alone is not sufficient to choose
it. Fit both, and check what k=4 actually does to the k=3 solution before
deciding.

In [ ]:
def fit_profiles(K, Z, comp):
    gm = GaussianMixture(n_components=K, covariance_type="diag", n_init=20, random_state=42)
    labels = gm.fit_predict(Z.values)
    c = comp.copy()
    c["profile"] = labels + 1
    return c, gm

c3, gm3 = fit_profiles(3, Z, comp)
c4, gm4 = fit_profiles(4, Z, comp)

for K, c in [(3, c3), (4, c4)]:
    print(f"\n{'='*55}\nK={K}\n{'='*55}")
    print("Profile sizes:", c["profile"].value_counts().sort_index().to_dict())
    print("\nProfile means:")
    print(c.groupby("profile")[indicators + ["CTE"]].mean().round(2))
    groups = [c.loc[c.profile == k, "CTE"].values for k in sorted(c.profile.unique())]
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"\nOmnibus ANOVA: F({len(groups)-1},{len(c)-len(groups)}) = {f_stat:.2f}, p = {p_val:.6f}")

## Step 3 — Tukey HSD: does every profile actually differ from every other?

An omnibus ANOVA only tells you *some* pair differs. If k=4 produces two
profiles that are statistically indistinguishable on CTE, that split isn't
earning its added complexity — it's fragmentation, not real separation.

In [ ]:
print("=== K=3 Tukey HSD on CTE ===")
print(pairwise_tukeyhsd(c3["CTE"], c3["profile"]))

print("\n=== K=4 Tukey HSD on CTE ===")
print(pairwise_tukeyhsd(c4["CTE"], c4["profile"]))

## Step 4 — How does k=4 relate to k=3? (cross-tabulation)

This shows whether k=4 splits a single k=3 profile into two, merges profiles
differently, or does something else entirely.

In [ ]:
crosstab = pd.crosstab(c3["profile"], c4["profile"], rownames=["K3"], colnames=["K4"])
print(crosstab)
print("""
Read this as: for each K=3 profile (row), how its members were redistributed
across the K=4 profiles (columns). A K=3 profile that maps cleanly onto one or
two K=4 profiles, where those K4 profiles are Tukey-distinct on CTE, indicates
a meaningful split. A K=3 profile whose members are dispersed into two K4
profiles that Tukey says do NOT differ on CTE indicates fragmentation.
""")

## Step 5 — Decision

On this dataset: every K=3 profile pair was Tukey-significant (all three
comparisons p<.01) — each of the three profiles is genuinely, statistically
distinct on the outcome. At K=4, one pair (the two profiles that K=3's largest
profile split into) was **not** Tukey-significant (p=.989) — meaning that
particular split added a group boundary without adding any real distinction in
critical thinking engagement. K=4 also isolated one small (~12% of the
sample), genuinely distinct high-performing subgroup with the highest CTE of
any profile at either k — a real, substantively interesting finding, just not
one that justifies making k=4 the primary model.

**Decision: retain k=3 as the primary solution** (every profile pair
Tukey-distinct; simpler; sufficient to support the paper's claim), and
**report k=4 as a sensitivity check**.

In [ ]:
K_FINAL = 3
comp["profile"] = c3["profile"]
comp.to_csv("composite_scores_with_profiles.csv")
print(f"Final K = {K_FINAL}. Saved profile assignments to composite_scores_with_profiles.csv")
print("\nFinal K=3 profile summary:")
print(comp.groupby("profile")[indicators + ["CTE"]].mean().round(2))

**What to look for in the final profiles:** the profile with the highest CTE
should be elevated in literacy and metacognition, not necessarily usage — if
the highest-usage profile also has the lowest or a middling CTE, that is
corroborating evidence for the structural model's finding, from a method that
never used a regression at all.

**Next notebook:** `06_ml_robustness.ipynb`